## OM 620 - Tools and Technologies for Business Analytics 

**Assignments 1 & 2 combined** , Cristina Rodriguez - Group 1 

## Introduction

Within this Jupyter Notebooks, I have combined Assignment 1 and 2 from OM 620 which uses the Dataset that can be found in the data folder. 

# Assignment 1  

For the strat of the assignment, we need to import pandas and numpy in order to be able to inspect and clean the data. We also need to import the dataset, which is in this case assigned to **transactions**.

In [2]:
import pandas as pd
import numpy as np
transactions = pd.read_csv("../data/transaction_data.zip")

## Question 1  - Data Frame formatting

Before starting any work on the data, we need to inspect it so that we know how the data is structured.

In [3]:
transactions.info()
transactions.sample(5)

<class 'pandas.DataFrame'>
RangeIndex: 760460 entries, 0 to 760459
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   sku_numb3r          757837 non-null  str    
 1   inventory_type      747418 non-null  str    
 2   stocking_type       745287 non-null  str    
 3   lead_time           760460 non-null  int64  
 4   unit_price          760460 non-null  float64
 5   Manufacturing Site  612982 non-null  str    
 6   division Code       707229 non-null  str    
 7   transaction_date    760460 non-null  str    
 8   order_quantity      760460 non-null  int64  
dtypes: float64(1), int64(2), str(6)
memory usage: 52.2 MB


,sku_numb3r,inventory_type,stocking_type,lead_time,unit_price,Manufacturing Site,division Code,transaction_date,order_quantity
698283,E352B076,FG,NaN,21,185.216527,CA-5A,94CF,2025-10-02,-2
114542,5FD708E6,FG,MTS,28,995.817050,NaN,B1E2,2023-06-15,87
694633,C1AC2891,FG,MTO,21,182.569776,CA-5A,9BA9,2025-09-27,27
342637,16C854A8,RM,MTO,28,1010.012178,JP-5C,82D0,2024-05-09,97
465937,D0FA322B,FG,MTO,28,1012.343464,JP-5C,8E41,2024-11-02,100


- Dataset structure  
The dataset has 760,460 observations, and 9 columns:  
**sku_numb3r** - stock keeing unit number  
**inventory_type** - types of inventory (FG: finished goods, WIP: work in progress, RM: raw material)  
**stocking_type** - stocking type policy (MTS: make to stock, MTO: make to order, ATO: assemble to order)  
**lead_time** - total lead time in days  
**unit_price** - unit price in US dollars  
**Manufacturing Site** - Manufacturing site ID starting with country code from USA state  
**division Code** - four character business division ID  
**transaction_date** - date of Transaction in Year-Month-Day format  
**order_quantity** - order quantity rounded up  

Based on the sample, we can see that the column names do not follow the same naming convention. This needs to be aligned with the metadata to allow easier use in the future.  
sku_numb3r - has a 3 instead of an e --> change to sku_number  
Manufacturing Site - has capitalization and a space --> change to manufacturing_site  
division Code - has capitalization and a space --> change to division_code

In [4]:
transactions.columns = ["sku_number", "inventory_type", "stocking_type", "lead_time", "unit_price", "manufacturing_site", "division_code", "transaction_date", "order_quantity"]
transactions.columns

Index(['sku_number', 'inventory_type', 'stocking_type', 'lead_time',
       'unit_price', 'manufacturing_site', 'division_code', 'transaction_date',
       'order_quantity'],
      dtype='str')

**transactions.columns** refers to all the columns in the transations data frame. using **=** we replace the existing columns with the column names in the new list, to match the metadata. we need to rename the columns in the same order that they are in currently.  
Then we inspect the columns to make sure the changes were applied. 

## Question 2  Data inspection

We use the **.describe()** function to inspect the numeric oddities in the dataset. 

In [5]:
transactions.describe()

,lead_time,unit_price,order_quantity
count,760460.000000,760460.000000,760460.000000
mean,25.775126,798.052454,79.865492
std,10.724936,374.638970,41.933496
min,14.000000,-1018.428051,-135.000000
25%,28.000000,981.057439,63.000000
50%,28.000000,995.994200,93.000000
75%,28.000000,1004.126123,109.000000
max,2800.000000,1024.337563,192.000000


Looking at the table, there are three variables to inspect - lead_time, unit_price and order_quantity.  
The main thing we are looking for is whether the minimum, maximum, and quartiles make sense compared with the rest of the values.  

- In the **lead_time** column, most of the values are tightly grouped, except for the maximum value of 2800, which is extremely far from the rest of the data, considering that 75% of the observations are exactly 28. This suggests that some of the values may have accidentally added zeroes.  
We will correct this by filtering the data to show lead times over 100 days. The value of 100 was chosen as a quick way to isolate values that are clearly far above the normal range.  

- In the **unit_price** column, the minimum value is a negative number. Unit price would normally represent the price of one unit, and should normally not be below zero. The rest of the distribution is between 981 and 1024, so the value seems to be similar to the normal prices, but the sign is negative.  
We will filter the data to show values below 0, and if these appear to be normal prices with an incorrect negative sign, they can be converted to positive values.  

- In the **order_quantity** column we have another negative value in the minimum row. Though suspicious, negative quantities could potentially represent returns, cancellations, corrections... We need to inspect these first, and then determine further action. 

- **Cleaning lead_time column**

In [6]:
transactions[transactions["lead_time"] > 100]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
58529,1141EA6F,FG,MTO,2800,985.744535,JP-F3,5A6B,2023-03-26,109
134978,D9BB7F45,WIP,MTS,2800,990.760849,JP-5C,EAEE,2023-07-15,134
363558,347857BA,RM,MTO,2800,1018.551873,CA-5A,FDCB,2024-06-08,98
479436,7809F5D1,FG,MTO,2100,202.728621,CA-5A,ED3D,2024-11-21,64
509017,3789F452,WIP,MTO,2100,187.510708,JP-B0,69A2,2025-01-03,31
521854,C318378C,FG,MTO,2800,1013.996493,PL-89,DB2F,2025-01-21,81
637832,3BEBF5F1,FG,MTO,2800,992.002568,JP-B1,5DF7,2025-07-07,116
638980,B0AB8C07,RM,MTS,2800,996.056173,CA-03,DB2F,2025-07-08,114
648717,B80052D1,WIP,MTS,2800,1001.608862,CA-5A,DFD7,2025-07-23,93
732265,257728D3,FG,MTO,2800,979.357293,NaN,390D,2025-11-20,92


The datatable shows 10 values that are far above the rest of the observations. We can safely assume that two zeroes were accidentally added to these values, and that the best course for cleaning this data would be to change these values as follows:  
2100 --> 21  
2800 --> 28

In [7]:
transactions["lead_time"] = transactions["lead_time"].replace({2100:21, 2800:28})
transactions.describe()

,lead_time,unit_price,order_quantity
count,760460.000000,760460.000000,760460.000000
mean,25.740497,798.052454,79.865492
std,4.768944,374.638970,41.933496
min,14.000000,-1018.428051,-135.000000
25%,28.000000,981.057439,63.000000
50%,28.000000,995.994200,93.000000
75%,28.000000,1004.126123,109.000000
max,28.000000,1024.337563,192.000000


From the dataset transactions, we access the "lead_time" column, and using the .replace() function, we change the values as defined above.  
Using the describe function, we verify that this change was successful, and we can see that the max lead time is now 28.  

- **Cleaning unit_price column**

In [8]:
transactions[transactions["unit_price"]<0]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
49984,2AE360E8,FG,MTO,28,-987.145612,CA-5A,EAEE,2023-03-14,116
56713,4E692D00,FG,MTO,28,-994.612343,JP-5C,69A2,2023-03-23,85
264658,3247A1B9,FG,MTO,28,-982.580274,CA-5A,8E41,2024-01-18,103
315652,5BD75D98,RM,MTS,14,-47.368801,JP-FB,69A2,2024-03-31,15
431105,5E02EF37,FG,MTO,28,-1018.428051,NaN,69A2,2024-09-12,108
434302,FA4AE7D0,FG,MTO,28,-993.553563,JP-5C,DCA7,2024-09-17,98
521860,1141EA6F,FG,MTO,28,-985.744535,CA-75,5DF7,2025-01-21,133
538857,9E119356,FG,MTO,28,-990.238172,NaN,DFD7,2025-02-14,106
645799,70F56985,WIP,MTS,28,-988.974884,CA-5A,5DF7,2025-07-18,134
715103,299299C2,RM,MTS,28,-1007.214144,CA-5A,5DF7,2025-10-26,106


The table shows another 10 values with incorrect formats. All of these values seem to be normal prices, with incorrect negative values. Instead of using the .replace() function like we previously did, the .abs() function will be used to concert these to positive values. If we had used the .replace() function, we would have had to type the 10 values and their replacements manually. 

In [9]:
transactions["unit_price"] = transactions["unit_price"].abs()
transactions.describe()

,lead_time,unit_price,order_quantity
count,760460.000000,760460.000000,760460.000000
mean,25.740497,798.076113,79.865492
std,4.768944,374.588567,41.933496
min,14.000000,28.824318,-135.000000
25%,28.000000,981.057439,63.000000
50%,28.000000,995.994200,93.000000
75%,28.000000,1004.126123,109.000000
max,28.000000,1024.337563,192.000000


Checking with the describe function, we can see that the replacement was successful. 

- **Inspecting order_quantity column**  

First, we inspect the suspicious negative values in order to determine how to proceed. 

In [10]:
transactions[transactions["order_quantity"] <0]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
7,2A29C5AC,RM,MTO,14,52.286677,JP-5C,8EBD,2023-01-01,-1
36,312E94A9,FG,MTO,14,54.699918,PL-E2,DCA7,2023-01-01,-11
40,A2193035,RM,MTS,14,47.717307,JP-5C,DCA7,2023-01-01,-9
71,8FF8A9B7,WIP,MTS,21,196.854536,JP-5C,5DF7,2023-01-01,-1
78,F8C4853C,WIP,MTS,21,207.425464,CA-5A,9BA9,2023-01-01,-16
...,...,...,...,...,...,...,...,...,...
760369,8D21AF3D,RM,MTS,21,197.485532,CA-75,CCE6,2025-12-30,-7
760398,BDB5C01B,FG,MTS,14,41.303224,JP-FB,DFD7,2025-12-30,-12
760404,35BD4EA5,WIP,MTS,14,55.694193,CA-5A,8EBD,2025-12-30,-2
760436,1D0E840A,FG,MTO,14,51.831299,CA-5A,FDAC,2025-12-30,-11


Since there are many entries with negative order quantity (39,107 rows), and there is no available information to establish they are data entry errors, these values will be retained. 

## Question 3 - Missing values

In order to determine how missing values need to be delat with, we need to use .isna().sum(), so that we get the total number of NaN values in each column. 

In [11]:
transactions.isna().sum()

sku_number              2623
inventory_type         13042
stocking_type          15173
lead_time                  0
unit_price                 0
manufacturing_site    147478
division_code          53231
transaction_date           0
order_quantity             0
dtype: int64

The columns with missing values are sku_number (2,623), inventory_type (13,042), stocking_type (15,173), manufacturing_site (147,478), and division_code (53,231). The columns lead_time, unit_price, transaction_date, and order_quantity had no missing values.

The missing values in the SKU and inventory classification columns need attention because these are important for identifying products and filtering the transactions for the subsequent safety stock analysis.  

A row without **sku** cannot be assigned to an inventory item, and the eventual purpose of this assignment is safety stock analysis by SKU, so these should be dropped using .dropna(subset = ["sku_number"]).  

Dropping rows with **missing inventory classification** is reasonable too because we have no way of knowing the missing inventory type, so we cannot randomly fill them with other values. We could fill them with "Unknown", but they would be removed later anyway when we filter specifically for FG in question 5.  The same logic applies to **missing stocking type**. We remove these three because we cannot answer questions such as: What product is this? Is it finished goods? Is it make to stock?

Missing manufacturing site and division information may also limit geographic and division-level comparisons. I will avoid assigning random product identifiers or inventory classifications because doing so could make the analysis inaccurate.

In [12]:
transactions_clean = transactions.dropna(subset=["sku_number"])
transactions_clean = transactions_clean.dropna(subset=["inventory_type"])
transactions_clean = transactions_clean.dropna(subset=["stocking_type"])

We used **transactions.dropna(subset=["sku_number"])** to drop the NA values from the column sku_number. Then, subset= is used because by default it would have dropped all of the NA values from the data. A new object is defined (transactions_cleaned) to store the cleaned data, and preserve the original data in transactions, in case it needs to be referenced in the future. The same code applies to inventory types and and stocking type. 

Missing **manufacturing site and division code**  
Almost 19.4% of the dataset has missing manufacturing site, so dropping these would remove a big portion of the dataset. These rows should be preserved so we should fill this NaN information with “Unknown”, and the same procedure should be followed for the division code, since these rows need to be perserved for the safety stock analysis.

In [13]:
transactions_clean["manufacturing_site"] = transactions_clean["manufacturing_site"].fillna("Unknown")
transactions_clean["division_code"] = transactions_clean["division_code"].fillna("Unknown")

In [14]:
transactions_clean.isna().sum()

sku_number            0
inventory_type        0
stocking_type         0
lead_time             0
unit_price            0
manufacturing_site    0
division_code         0
transaction_date      0
order_quantity        0
dtype: int64

Now this is the new sum of NA values for the dataset.

## Question 4 - Extracting country values  

In order to understand how country vakues are stored, we need to examine the manufacturing_site column and it's values.

In [15]:
transactions_clean["manufacturing_site"].value_counts()

manufacturing_site
CA-5A      142386
JP-5C      142028
Unknown    141526
CA-8A      141317
JP-60       28536
US-99       28479
PL-89       13996
JP-FB        7212
US-04        7136
CA-75        7118
US-5A        7115
PL-E2        7097
JP-B0        7095
JP-B1        7058
CA-0B        7031
US-61        7017
PL-B0        6992
CA-03        6969
PL-55        6950
JP-F3        6833
Name: count, dtype: int64

From the result, we can see that the code for the manufacturing site is consisted of country codes from USA(US), Canada(CA), Poland(PL) and Japan (JP), and manufacturing site ID's (eg. 5A, 99...). First, we need to create  new column to store the extracted country codes, and assign "Unknown" to the whole column to start with.  

In [16]:
transactions_clean["country_code"] = "Unknown"

Now that we have a new column, we can extract the country codes from manufacturing_site to country_code.

In [17]:
transactions_clean.loc[transactions_clean["manufacturing_site"].str.startswith("US"), "country_code"] = "US"
transactions_clean.loc[transactions_clean["manufacturing_site"].str.startswith("CA"), "country_code"] = "CA"
transactions_clean.loc[transactions_clean["manufacturing_site"].str.startswith("PL"), "country_code"] = "PL"
transactions_clean.loc[transactions_clean["manufacturing_site"].str.startswith("JP"), "country_code"] = "JP"

**transactions_clean["manufacturing_site"]** selects the manufacturing site column from the dataset.  
**.str.startswith("US")** checks wether each value in the column starts with the defined letters. This returns boolean values for every row.   
**transactions_clean.loc[]** is used to select particular rows and culumns in a DF. In this code, rows is *transactions_clean["manufacturing_site"].str.startswith("US")* and the column is only the *country_code* column. Therefore, the whole code asks to find all rows where the manufacturing site strats with US and select their country values. Then **= "US"** assigns US to the selected cells.

In [18]:
transactions_clean["country_code"].value_counts()

country_code
CA         304821
JP         198762
Unknown    141526
US          49747
PL          35035
Name: count, dtype: int64

We count the values in the new column to check that the code was successfully applied. There are still unknown values in the column, but that is due to the unknown values in the manufacturing site column. 

## Question 5 - Helpful Dimensional Information  

Since we ultimately want to calculate safety stock, we need to focus on finished goods designated as "make to stock." Create a filtered subset of your transaction dataset and write appropriate code to answer the following questions (for the filtered data).  

In the dataset **FG = Finished goods and MG = Make to stock**. Finished goods is in the inventory type column, and Make to stock is in the stocking type column. 

In [19]:
filtered_data = transactions_clean[(transactions_clean["inventory_type"] == "FG") & (transactions_clean["stocking_type"] == "MTS")]  
filtered_data.head()

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country_code
2,036A78DC,FG,MTS,28,999.652063,Unknown,8E41,2023-01-01,115,Unknown
3,E90CE574,FG,MTS,28,1010.156998,Unknown,E358,2023-01-01,119,Unknown
6,708D6AF0,FG,MTS,28,1003.935380,CA-8A,Unknown,2023-01-01,99,CA
9,5A83CDC4,FG,MTS,28,1005.647070,JP-B1,DFD7,2023-01-01,98,JP
10,C8E27FB5,FG,MTS,28,994.444643,US-99,9BA9,2023-01-01,98,US


Since we are looking at finished goods that are made to stock, the subset should only show rows for these two conditions. We define a new object called filtered_data where only transactions classified as FG and MTS are stored.  
**(transactions_clean["inventory_type"] == "FG")**  checks wether inventory type = finished goods  
**(transactions_clean["stocking_type"] == "MTS")** checks if stocking type = make to stock  
This creates true or false values, and the operator **&** requires both conditions to be true in order for a row to be included. 

### How many unique SKUs are we working with?

In [20]:
filtered_data["sku_number"].value_counts()

sku_number
BDB5C01B    5395
CF33D156    3998
2249FBC0    3776
DB97FEC7    3751
8FF8A9B7    3685
            ... 
70F56985      71
1C59FAC9      71
7C7D6696      68
1141EA6F      67
1FAC49D7      67
Name: count, Length: 360, dtype: int64

From the filtered data, we select the sku number column and call the value counts function. This provides us with a Series of the sku number values and their counts in the data. The length of the series is 360, which means there are **360 unique SKUs**.

### How many unique Manufacturing Sites are we working with?

In [21]:
filtered_data["manufacturing_site"].value_counts()

manufacturing_site
CA-5A      50946
Unknown    50321
JP-5C      50247
CA-8A      50244
JP-60      10134
US-99      10092
PL-89       5024
US-04       2623
JP-FB       2553
PL-E2       2550
PL-B0       2522
JP-B1       2517
JP-B0       2513
CA-75       2492
US-5A       2488
CA-0B       2473
PL-55       2465
CA-03       2434
US-61       2410
JP-F3       2349
Name: count, dtype: int64

From the filtered datam we get a Series that lists the uique manufacturing sites.

In [22]:
filtered_data["manufacturing_site"].value_counts().count()

np.int64(20)

To count the length of the series, we apply the .count() function on top of that and the result is **20 unique manufacturing sites**, including unknown values, so . 

### How many Divisions are we working with?

In [23]:
filtered_data["division_code"].value_counts().count()

np.int64(55)

We apply the same logic for the division codes column, and the result is **55 unique divisions**, again including unknown values.

### What are the top and bottom 10 transactions by order quantity?  

For the bottom 10, we sort the values of the order quantity column, followed by .head(10) to show the first 10 observations. The **.sort_values** function automatically sorts by ascending, so the orders with the lowest order quantity will show up on top. 

In [24]:
filtered_data.sort_values("order_quantity").head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country_code
411026,CF33D156,FG,MTS,28,1021.354856,Unknown,A866,2024-08-15,-95,Unknown
75402,7747965D,FG,MTS,28,980.002831,CA-5A,3D40,2023-04-20,-95,CA
41032,00D0DF3B,FG,MTS,28,1010.057470,CA-8A,ACEE,2023-03-01,-93,CA
238079,A093EB12,FG,MTS,28,998.281566,JP-5C,8E41,2023-12-11,-88,JP
507799,04CD4C8F,FG,MTS,21,212.250683,Unknown,69A2,2025-01-01,-63,Unknown
191297,6BCB8698,FG,MTS,21,198.135265,JP-60,9BA9,2023-10-04,-58,JP
746699,CCD3552E,FG,MTS,21,208.118486,US-99,DB2F,2025-12-11,-58,US
581208,19FB77E3,FG,MTS,21,199.756981,US-61,9BA9,2025-04-16,-54,US
753183,8760340C,FG,MTS,21,206.923468,JP-60,69A2,2025-12-20,-49,JP
625425,EF035773,FG,MTS,21,195.176706,PL-B0,Unknown,2025-06-19,-47,PL


All the values of the bottom 10 transactions by order quantity are negative, so -95 shows up on top.

To sort the data by descending, so that the biggest values show up on top, we need to add **ascending= False** to the sort values function. 

In [25]:
filtered_data.sort_values("order_quantity", ascending = False).head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country_code
332825,361B3A3C,FG,MTS,28,994.572406,CA-5A,DFD7,2024-04-25,189,CA
417833,EF1EBAE2,FG,MTS,28,999.393166,Unknown,5DF7,2024-08-24,187,Unknown
566007,9DD9199E,FG,MTS,28,1009.445290,CA-75,4BA1,2025-03-25,187,CA
196131,4626B062,FG,MTS,28,987.536273,JP-5C,CCE6,2023-10-11,186,JP
122621,4626B062,FG,MTS,28,987.536273,CA-8A,E358,2023-06-27,185,CA
62719,BDB5C01B,FG,MTS,28,992.623768,CA-8A,CCE6,2023-04-01,181,CA
165676,8FF8A9B7,FG,MTS,28,991.330046,CA-5A,EAEE,2023-08-28,181,CA
455991,E1E201E9,FG,MTS,28,1012.145121,CA-5A,8E41,2024-10-18,180,CA
722303,2077D3E8,FG,MTS,28,1002.666038,US-99,EAEE,2025-11-05,179,US
575090,0BC11363,FG,MTS,28,1018.203896,JP-5C,69A2,2025-04-07,179,JP


### What are the top and bottom 10 transactions by total sales value?  

Since sales is not a feature in this dataset, we need to calculate the sales by multyplying the columns unit price and order quantity. We also need to create a new column to store these values. 

In [26]:
filtered_data["total_sales_value"] = (filtered_data["order_quantity"] * filtered_data["unit_price"])
filtered_data.sample(5)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country_code,total_sales_value
207437,7B34BF2E,FG,MTS,28,984.802496,CA-8A,3D40,2023-10-27,103,CA,101434.657087
458119,CF33D156,FG,MTS,28,1001.837273,Unknown,DB2F,2024-10-21,110,Unknown,110202.100081
653795,8A337DA9,FG,MTS,21,207.341684,Unknown,8E41,2025-07-30,45,Unknown,9330.375787
228204,9A880BF6,FG,MTS,28,1010.617191,JP-5C,8E41,2023-11-26,97,JP,98029.867488
576776,0DE7DD86,FG,MTS,28,1001.331916,CA-8A,5DF7,2025-04-10,100,CA,100133.191551


We use .sample(5) to show 5 rando rows in the dataset and check that the new column was added with the right values.  
From here, we can apply the same code as the previous question to show the top 10 and bottom 10 orders by sales value.  

Top 10:

In [27]:
filtered_data.sort_values("total_sales_value", ascending = False).head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country_code,total_sales_value
566007,9DD9199E,FG,MTS,28,1009.445290,CA-75,4BA1,2025-03-25,187,CA,188766.269262
332825,361B3A3C,FG,MTS,28,994.572406,CA-5A,DFD7,2024-04-25,189,CA,187974.184791
417833,EF1EBAE2,FG,MTS,28,999.393166,Unknown,5DF7,2024-08-24,187,Unknown,186886.521965
196131,4626B062,FG,MTS,28,987.536273,JP-5C,CCE6,2023-10-11,186,JP,183681.746686
122621,4626B062,FG,MTS,28,987.536273,CA-8A,E358,2023-06-27,185,CA,182694.210413
575090,0BC11363,FG,MTS,28,1018.203896,JP-5C,69A2,2025-04-07,179,JP,182258.497461
455991,E1E201E9,FG,MTS,28,1012.145121,CA-5A,8E41,2024-10-18,180,CA,182186.121808
608057,E1E201E9,FG,MTS,28,1012.145121,CA-5A,17D3,2025-05-25,178,CA,180161.831566
62719,BDB5C01B,FG,MTS,28,992.623768,CA-8A,CCE6,2023-04-01,181,CA,179664.901949
722303,2077D3E8,FG,MTS,28,1002.666038,US-99,EAEE,2025-11-05,179,US,179477.220864


Bottom 10:

In [28]:
filtered_data.sort_values("total_sales_value").head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country_code,total_sales_value
411026,CF33D156,FG,MTS,28,1021.354856,Unknown,A866,2024-08-15,-95,Unknown,-97028.711351
41032,00D0DF3B,FG,MTS,28,1010.057470,CA-8A,ACEE,2023-03-01,-93,CA,-93935.344750
75402,7747965D,FG,MTS,28,980.002831,CA-5A,3D40,2023-04-20,-95,CA,-93100.268967
238079,A093EB12,FG,MTS,28,998.281566,JP-5C,8E41,2023-12-11,-88,JP,-87848.777852
507799,04CD4C8F,FG,MTS,21,212.250683,Unknown,69A2,2025-01-01,-63,Unknown,-13371.793040
746699,CCD3552E,FG,MTS,21,208.118486,US-99,DB2F,2025-12-11,-58,US,-12070.872198
191297,6BCB8698,FG,MTS,21,198.135265,JP-60,9BA9,2023-10-04,-58,JP,-11491.845367
581208,19FB77E3,FG,MTS,21,199.756981,US-61,9BA9,2025-04-16,-54,US,-10786.876960
753183,8760340C,FG,MTS,21,206.923468,JP-60,69A2,2025-12-20,-49,JP,-10139.249950
746931,1144D57C,FG,MTS,21,214.758781,Unknown,Unknown,2025-12-11,-46,Unknown,-9878.903907


## References 

No outside refrences were used in this assignment. 

## Assignment 2 - Safety Stock Calculations 

In Assignment 2 we now calculate how much safety stock the company should keep for each SKU. We ended up having to filter through the cleaned data for the correct SKUs and aggregate it by SKUs, calculate the minimum, maximum, average and median, variance and standard deviated of order quantity as well as the average lead time. We also use the data to calculate safety stock for each SUKU at 75%, 90% and 95% service levels with the forumula that was given. Lastly we looked at the safety stock values at the 95% service levels to find the SKUs with the largest and smallest safety stock and the average stock across the SKUs. 

## Introduction

In the first assignment, we inspected and cleaned the transaction dataset so it could be used for a safety stock analysis. The dataset had 760,460 transactions and 9 columns. The first step was renaming some of the columns (sku_numb3r, Manufacturing Site and division Code) so that they matched the metadata and followed the same naming convention.

When inspecting the numeric columns, we found a few problems. Some lead times had two extra zeroes added (2100 and 2800 instead of 21 and 28), so we corrected them. A few unit prices were negative but otherwise looked like normal prices, so we changed them to positive values. The order_quantity column also had negative values. In Assignment 1 we kept them, but based on the feedback we received, these are most likely data entry errors with an incorrect negative sign so in this assignment we convert them to positive values with `.abs()`.

For missing values, we dropped the rows with no SKU number, inventory type or stocking type, and filled missing manufacturing sites and division codes with "Unknown". The feedback on Assignment 1 pointed out that filling with "Unknown" is the safer option for all columns, because the data has several layers of information (SKU, type, site, division) and dropping a row because one of them is missing removes the information in the others too. In this assignment we will fill all missing values with "Unknown". 

Lastly, we filtered the data to keep only finished goods (FG) that are make to stock (MTS), because these are the items safety stock can be calculated for. In this filtered data we found **360 unique SKUs**, 20 manufacturing sites and 55 divisions (both counts include "Unknown").

A few of these findings matter for this assignment:
- The analysis is again done on the FG and MTS subset, so we should end up with safety stock values for 360 SKUs.
- The lead times are now only 14, 21 or 28 days, which matters because lead time is part of the safety stock formula.
- The negative order quantities are now converted to positive values. This is important because the standard deviation of order quantity is part of the safety stock formula.

In this assignment we will calculate how much safety stock should be kept for each SKU, so that the company can handle changes in demand. To do this, we will:
1. Filter the cleaned data for the right SKUs and aggregate it by SKU, calculate the minimum, maximum, average, median, variance and standard deviation of order quantity, along with the average lead time.
2. Use this data to calculate safety stock for each SKU at service levels of 75%, 90% and 95%, using the provided formula.
3. Look at the safety stock values at the 95% service level to find the SKUs with the largest and smallest safety stock, and the average safety stock across all SKUs.

## Data cleaning

Since this is a new notebook, we first need to repeat the cleaning steps from Assignment 1 so that we work with the same cleaned data. We rename the columns to match the metadata, correct the lead times that had extra zeroes and change the negative unit prices and order quantities to positive values. For missing values, we change our approach from Assignment 1. Instead of dropping rows with a missing SKU number, inventory type or stocking type, we fill all missing values with "Unknown", as suggested in the feedback. This way no rows are lost. Rows with "Unknown" values that aren't suitable for the safety stock calculation are removed later, when we filter the data in Question 1.   

The lead time fix is important here, because lead time is used directly in the safety stock formula, and values like 2800 would make the results too high. Unit price isn't used in this assignment, but we still fix it so the dataset is consistent with the first assignment. Some steps from Assignment 1 are not repeated, because they aren't needed for the safety stock calculation. We don't extract the country codes or calculate the total sales value, since this assignment only needs the SKU number, order quantity and lead time.  

The cleaned dataset could have been exported from the previous assignment, but the instructions were unclear if we can do that, so we just repeat the same steps. 

In [30]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.stats.mstats import mquantiles

transactions = pd.read_csv("../data/transaction_data.zip")

In [ ]:
# Renaming columns
transactions.columns = ["sku_number", "inventory_type", "stocking_type", "lead_time", "unit_price",
                        "manufacturing_site", "division_code", "transaction_date", "order_quantity"]

# Fixing outliers from the columns
transactions["lead_time"] = transactions["lead_time"].replace({2100:21, 2800:28})
transactions["unit_price"] = transactions["unit_price"].abs()

#Making negative values in order_quantity column to positive based on assignment 1 feedback
transactions["order_quantity"] = transactions["order_quantity"].abs()

# Filling all missing values with "Unknown" based on assignment 1 feedback
transactions_clean = transactions.fillna("Unknown")

transactions_clean.sample(5)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
206295,AAFE52B8,RM,MTS,21,201.965709,CA-5A,CF10,2023-10-26,21
20290,B064F6FF,RM,MTO,28,996.767395,JP-5C,BE13,2023-01-30,119
544513,DB97FEC7,FG,MTS,28,993.970520,CA-5A,CCE6,2025-02-23,74
267925,376B60F0,FG,MTS,21,199.535442,JP-60,DCA7,2024-01-22,51
486575,25D051CE,FG,MTO,21,209.386113,US-99,5DF7,2024-12-01,39


## Question 1 — Required Transformation
*Create a transformed dataset to filter for appropriate SKUs that are also aggregated with the following statistics: minimum, maximum, average, median, variance, and standard deviation of quantity, along with average lead time.*

The safety stock formula needs the standard deviation of demand and the lead time for each SKU, but at the moment our data has one row for every transaction. That means we first need to group the transactions by SKU and summarise them, so that each SKU has its own row with the statistics we need.

We chose `.groupby()` together with `.aggregate()` to do this, which also enables us to calculate all the required statistics in one step, instead of making a separate table for each one and merging them together. We only use the FG and MTS data, because make-to-stock finished goods are the only items we can calculate safety stock for. We also add a third condition to filter out the "Unknown" SKU because otherwise `.groupby("sku_number")` would treat all the unknown SKUs as one product and calculate a safety stock for it, even though these rows most likely belong to many different products.

In [ ]:
#We filter to see only FG and MTS stock that also do not have an "Unknown" SKU
filtered_data = transactions_clean[(transactions_clean["inventory_type"] == "FG") & (transactions_clean["stocking_type"] == "MTS") & 
                                   (transactions_clean["sku_number"] != "Unknown")]

sku_stats = (
    filtered_data
    .groupby("sku_number")
    .aggregate({"order_quantity": ["min", "max", "mean", "median", "var", "std"], "lead_time": "mean"})
)
sku_stats.columns = ["min_quantity", "max_quantity", "avg_quantity", "median_quantity", "var_quantity", "std_quantity", "avg_lead_time"]
sku_stats

,min_quantity,max_quantity,avg_quantity,median_quantity,var_quantity,std_quantity,avg_lead_time
sku_number,,,,,,,
00D0DF3B,35,161,99.988852,101.0,393.366301,19.833464,28.000000
01083EC9,0,82,23.020997,20.0,249.989032,15.811041,21.000000
036A78DC,0,163,92.307971,98.0,1021.387756,31.959158,26.862319
0469CB75,0,36,8.934959,8.0,46.730446,6.835967,14.000000
047A0938,20,159,100.031720,100.0,405.407122,20.134724,28.000000
...,...,...,...,...,...,...,...
F9ED9C68,0,38,12.147239,11.0,68.706582,8.288943,28.000000
FACEC76F,0,29,8.147727,7.0,36.104363,6.008691,14.000000
FAF5C442,0,87,20.793792,17.0,231.350717,15.210218,19.711752


- For **order_quantity**, we give a list of calculations instead of just one: "min" (minimum), "max" (maximum), "mean" (average), "median", "var" (variance) and "std" (standard deviation). This way all six statistics are calculated in one step instead of writing a separate line for each.
- For **lead_time**, we only need the "mean". Some SKUs have more than one lead time value in the data, so the average gives us one lead time per SKU that we can use in the safety stock formula.

We use **sku_stats.columns = [...]** to rename the calculation columns. The new names have to be in the same order as the calculations in the aggregate function.

## Question 2- Safety Stock Calculation
*Use the transformed data from Question 1 to calculate three safety stock values for each SKU for the service level coefficients of 75%, 90%, and 95%. You will need to find the z-score of the given service level coefficients, for which you can use the ppf function of SciPy.Stat module of the SciPy package, or simply use the numeric values from any standard normal distribution function.*

In this question, we use the transformed data from Question 1 to calculate three safety stock values for each SKU, for service level coefficients of 75%, 90% and 95%. As explained in the assignment instructions, safety stock is calculated with the following formula:

**Safety stock = z × o × √L**

- **z** is the z-score of the service level coefficient (how many standard deviations of extra stock we need to keep above average demand, so that we can fill orders as often as the service level promises).
- **o** is the standard deviation of demand, which is the std_quantity column we calculated in Question 1,
- **L** is the lead time in days, which is the avg_lead_time column from Question 1.

The standard deviation and lead time are already in our data, but we still need the z-scores. To find them we use the **SciPy** package, as suggested in the assignment instructions (SciPy Developers, n.d. -a).

In [ ]:
z_75 = norm.ppf(0.75)
z_90 = norm.ppf(0.90)
z_95 = norm.ppf(0.95)
z_75, z_90, z_95

(np.float64(0.6744897501960817),
 np.float64(1.2815515655446004),
 np.float64(1.6448536269514722))

To find the z-scores, we use the `norm` object from the `scipy.stats`, which we imported in the set-up cell. **norm** is for the normal distribution. If we don't give it a mean or standard deviation, it uses the **standard** normal distribution, which has a mean of 0 and a standard deviation of 1. 

The function we use is `.ppf()`, (percent point function) (Tutorials Point, n.d.). If we want to have enough stock 95% of the time we need to answer what demand level is higher than 95% of days.

`norm.ppf(0.95)` tells us how high we need to go so that 95% of the normal distribution will be below the cutoff. The answer is the z-score for the 95% service level.

The result is about **1.645**, which means we need to keep enough stock to cover demand up to 1.645 standard deviations above the average. This way we will have enough safety stock for 95% of days and we would only run out on 5% of days that have really high demand. 

The same logic applies to the other two service levels. `norm.ppf(0.75)` gives about **0.674** and `norm.ppf(0.90)` gives about **1.282**. The higher the service level, the higher the z-score. If we want to be more sure that we won't run out of stock, we need a bigger stock above average demand.

We save the results in **z_75**, **z_90** and **z_95** so we can use them in the formula. 

The z-score goes up as the service level goes up: the more certain we want to be that we won't run out of stock, the more standard deviations of extra stock we need. Now that we have the z scores, and we have other parts of the formula already calculated in the `sku_stats` table, we can use it to calculate the safety stock. 

In [ ]:
sku_stats["safety_stock_75"] = z_75 * sku_stats["std_quantity"] * np.sqrt(sku_stats["avg_lead_time"])
sku_stats["safety_stock_90"] = z_90 * sku_stats["std_quantity"] * np.sqrt(sku_stats["avg_lead_time"])
sku_stats["safety_stock_95"] = z_95 * sku_stats["std_quantity"] * np.sqrt(sku_stats["avg_lead_time"])
sku_stats.head()

,min_quantity,max_quantity,avg_quantity,median_quantity,var_quantity,std_quantity,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95
sku_number,,,,,,,,,,
00D0DF3B,35,161,99.988852,101.0,393.366301,19.833464,28.000000,70.786909,134.497334,172.625460
01083EC9,0,82,23.020997,20.0,249.989032,15.811041,21.000000,48.870353,92.855196,119.178354
036A78DC,0,163,92.307971,98.0,1021.387756,31.959158,26.862319,111.722959,212.277106,272.454716
0469CB75,0,36,8.934959,8.0,46.730446,6.835967,14.000000,17.251996,32.779330,42.071814
047A0938,20,159,100.031720,100.0,405.407122,20.134724,28.000000,71.862125,136.540280,175.247552


In this code, we calculate the safety stock for each SKU using the formula, and we save the results in three new columns, one for each service level.

- **z_75**, **z_90** and **z_95** are the z-scores we found. Each one is a single number, so it is the same for every SKU.
- `sku_stats["std_quantity"]` selects the whole standard deviation column.
- `np.sqrt(sku_stats["avg_lead_time"])` takes the square root of the average lead time column. We use the NumPy function `np.sqrt()` to calculate the square root of the average lead time. 

Pandas does the calculation row by row, so each SKU uses its own standard deviation and lead time and gets its own safety stock value. 

The table shows that the safety stock grows as the service level goes up. For example, SKU 00D0DF3B needs about 70 units for 75%, 134 units for 90% and 172 units for 95%. Going from 75% to 95% more than doubles the safety stock, so reaching a higher service level is more expensive when it comes to inventory.  

Variability could also affect safety stock because uncertain demand requires more safety stock to be kept to avoid stockouts when demand is high. 

## Question 3 — Safety Stock Distribution

*Now that we have the safety stock values, write appropriate code to answer the following questions. For this question, you may focus on the 95% service level coefficient.*

*Which SKU has the largest safety stock?*  
*Which SKU has the smallest safety stock?*  
*What is the average safety stock value across all SKUs?*  

### Largest safety stock  

To see the item with the largest safety stock for the 95% service level coefficient, we need to sort that column in the table we created from largest to smallest, and only display the top result. 

In [ ]:
sku_stats.sort_values("safety_stock_95", ascending=False).head(1)

,min_quantity,max_quantity,avg_quantity,median_quantity,var_quantity,std_quantity,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95
sku_number,,,,,,,,,,
A4287F10,0,171,81.038513,93.0,1696.831554,41.192615,26.215139,142.25607,270.29097,346.914705


The SKU with the largest safety stock at the 95% service level is **A4287F10**, with a safety stock of about **347 units**.

It has the largest safety stock because it has high demand variability and a long lead time. Its standard deviation is 41.19, the highest among all SKUs, and its order quantities range a lot. Its average lead time is about 26 days, close to the maximum of 28, so it takes a long time to restock and the safety stock has to cover a long period of time. This SKU also has high demand in general, with an average order quantity of about 81 units and a median of 93. Since demand is both high and unpredictable, the company would need to keep a large amount of extra stock for this product to meet orders 95% of the time.  

We compare with other top 5 to look at all the measures and determine why this has such high safety stock.

In [ ]:
sku_stats.sort_values("std_quantity", ascending=False).head(5)

,min_quantity,max_quantity,avg_quantity,median_quantity,var_quantity,std_quantity,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95
sku_number,,,,,,,,,,
A4287F10,0,171,81.038513,93.0,1696.831554,41.192615,26.215139,142.256070,270.290970,346.914705
58023D4F,0,165,79.365556,91.0,1623.838683,40.296882,26.430544,139.733279,265.497588,340.762465
EAEE18E6,0,164,83.595689,96.0,1609.696055,40.121018,26.447866,139.169034,264.425507,339.386464
A093EB12,0,166,81.193206,94.0,1539.376844,39.234893,25.874735,134.612621,255.768179,328.274904
361B3A3C,0,189,85.367415,96.0,1531.183775,39.130343,25.645485,133.657850,253.954085,325.946539


All the top SKUs share varying order quantity and a long average lead time. This means that safety stock has to cover longer periods of time for an uncertain demand, which is why the safety stock is so high. 

We can also look at how these SKUs compare with others. 

In [ ]:
sku_stats[["std_quantity", "avg_lead_time", "avg_quantity"]].describe()

,std_quantity,avg_lead_time,avg_quantity
count,360.000000,360.000000,360.000000
mean,13.107914,19.667821,35.658702
std,8.526981,6.195549,38.708815
min,5.690555,14.000000,7.560000
25%,6.669749,14.000000,8.971138
50%,7.620715,14.000000,10.332843
75%,19.706724,27.386968,91.148475
max,41.192615,28.000000,101.650199


A4287F10 has the highest standard deviation of all 360 SKUs (41.19), which is more than three times the average of 13.1. Its average lead time (26.2 days) and average order quantity (81.0 units) are also well above the average across all SKUs (19.7 days and 35.7 units).

### Smallest safety stock  

We use the same code as the previous question, but remove `ascending = False` so that the values are sorted from smallest to largest. We only view the top result to answer the question: Which SKU has the smallest safety stock?

In [ ]:
sku_stats.sort_values("safety_stock_95").head(1)

,min_quantity,max_quantity,avg_quantity,median_quantity,var_quantity,std_quantity,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95
sku_number,,,,,,,,,,
83AE2839,0,33,7.868132,7.0,32.382418,5.690555,14.0,14.361308,27.286934,35.0224


The SKU with the smallest safety stock at the 95% service level is **83AE2839**, with a safety stock of about **35 units**.

It has the smallest safety stock for the opposite reasons to A4287F10. Its standard deviation (5.69) is the lowest of all 360 SKUs, as the minimum in the describe table shows, so its demand is the most stable. Its lead time is 14 days, the shortest in the data, so it can be restocked quickly and the safety stock only needs to cover a short period.

This SKU also has low demand, with an average order quantity of about 8 units and a median of 7. Its safety stock of 35 units is about four and a half times its average order quantity, because even small orders still vary from 0 to 33 units.

### Average safety stock across all SKUs  

To calculate this we use the `.mean()` function on the `safety_stock_95` column. 

In [ ]:
sku_stats["safety_stock_95"].mean()

np.float64(102.02559297184781)

In [ ]:
sku_stats["safety_stock_95"].describe()

count    360.000000
mean     102.025593
std       78.827074
min       35.022400
25%       41.048831
50%       49.145774
75%      171.522346
max      346.914705
Name: safety_stock_95, dtype: float64

The average safety stock across all 360 SKUs at the 95% service level is about **102 units**.

However, using `.describe()` on the `safety_stock_95` column, we can see that the median is only about 49 units, and 75% of SKUs need around 172 units or less. The mean is much higher than the median because a smaller group of SKUs with high demand variability and long lead times pull the average up.

This shows that the SKUs fall into very different groups. In practice, it is best to look at safety stock for each SKU separately instead of using one average value for all products.

## Side Quest - Non-parametric safety stock

The safety stock formula we used in Question 2 is one of the most common methods in practice, but it assumes that the demand for each SKU follows a normal distribution. Real demand is often highly variable. In this section, we calculate the safety stock again with a non-parametric method, which means it doesn't assume any specific distribution, and compare the results with the normal formula.

Normal stock covers the average demand, and safety stock is the extra stock on top of that for days when demand is higher than average. So safety stock tells us how far above the average demand do we need to be ready for days with higher demand. 

For a 95% service level, we want a cut-off that demand stays below on 95% of days, and the safety stock is the gap between the average demand and this cut-off. If demand is normally distributed, the 95% cut-off is always 1.645 standard deviations above the average.

Before making the calculation, we researched quantiles and empirical quantiles. A quantile (or percentile) is a cut-off point in ordered data, where a certain share of the values falls below it. For example, 70% of the data falls below the 70th percentile (NIST/SEMATECH, n.d.). An **empirical quantile** is a quantile that we find by looking at the actual data, instead of calculating it from a formula that assumes a certain distribution. 

Instead of guessing the cut-off from the mean and standard deviation, we find it directly in the real orders of each SKU, and then calculate the same gap between the cut-off and the average demand. This gap replaces **z × o**. Both measure how far above the average the cut-off is, but one assumes a normal distribution and the other uses the real data. The lead time still works the same way, because the safety stock has to cover the whole time it takes to restock, so we keep multiplying by the square root of the lead time, like in the formula from the assignment instructions. 

**Non-parametric safety stock = (empirical quantile − average demand) × √L**

To find the empirical quantiles, we use the **mquantiles** function from the **scipy.stats.mstats** module, as suggested in the assignment instructions (SciPy Developers, n.d.-b).

### Calculating empirical quantiles  

In order to calculate the empirical quantiles for each SKU, we need to create our own functions that look at each SKU and give back its quantile. We need these functions because `.aggregate()` doesn't have a built-in option for quantiles like it does for "mean" or "std". We need to import the package that allows us to use this formula, so we add it to the import cell at the beginning of the notebook. `from scipy.stats.mstats import mquantiles` 

In [ ]:
# Functions that give the 75%, 90% and 95% empirical quantile of a column
def quantile_75(x):
    return mquantiles(x, prob=[0.75])[0]

def quantile_90(x):
    return mquantiles(x, prob=[0.90])[0]

def quantile_95(x):
    return mquantiles(x, prob=[0.95])[0]

Using `def` we start creating a new function. We choose the names `quantile_75`, `quantile_90` and `quantile_95` to represent the results.  

We use `x` as a placeholder for any data that we use the function on (which will be all the order quantities of one SKU).  

As per the assignment instructions we use `mquantiles(x, prob=[0.75/0.90/0.95])` to calculate the corresponding empirical quantiles of `x`. `[0]` takes the first answer in the list because the function returns its answer as a list. 

Since we want to calculate the quantiles of order_quantity for each SKU, we use the `groupby` and `aggregate` functions. We group the filtered data by `sku_number` and then apply our quantile functions to the `order_quantity` column.

In [ ]:
# Calculating the quantiles of order quantity for each SKU
sku_quantiles = (
    filtered_data
    .groupby("sku_number")
    .aggregate({"order_quantity": [quantile_75, quantile_90, quantile_95]})
)
sku_quantiles.columns = ["q75_quantity", "q90_quantity", "q95_quantity"]
sku_quantiles.head()

,q75_quantity,q90_quantity,q95_quantity
sku_number,,,
00D0DF3B,113.0,124.0,132.00
01083EC9,33.3,46.0,52.54
036A78DC,111.0,125.0,133.00
0469CB75,13.0,19.0,22.00
047A0938,114.0,125.0,134.00


The quantiles are not the safety stock themselves, but cut-offs in the demand. For example, q95_quantity tells us the order size that 95% of the orders don't go above. To get the safety stock, we still need to subtract the average demand (because normal stock already covers that) and multiply by the square root of the lead time, which we do in the next step. We merge the sku_quantiles to our sku_stats tables for further calculations. 

In [ ]:
sku_stats = pd.merge(sku_stats, sku_quantiles, on = "sku_number")
sku_stats.head()

,min_quantity,max_quantity,avg_quantity,median_quantity,var_quantity,std_quantity,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95,q75_quantity,q90_quantity,q95_quantity
sku_number,,,,,,,,,,,,,
00D0DF3B,35,161,99.988852,101.0,393.366301,19.833464,28.000000,70.786909,134.497334,172.625460,113.0,124.0,132.00
01083EC9,0,82,23.020997,20.0,249.989032,15.811041,21.000000,48.870353,92.855196,119.178354,33.3,46.0,52.54
036A78DC,0,163,92.307971,98.0,1021.387756,31.959158,26.862319,111.722959,212.277106,272.454716,111.0,125.0,133.00
0469CB75,0,36,8.934959,8.0,46.730446,6.835967,14.000000,17.251996,32.779330,42.071814,13.0,19.0,22.00
047A0938,20,159,100.031720,100.0,405.407122,20.134724,28.000000,71.862125,136.540280,175.247552,114.0,125.0,134.00


### Non-parametric safety stock 

We add three new columns to the sku_stats table which calculate the non parametric safety stock by subtracting the average demand from the calculated quantiles. This is how far above average demand the empirical quantile is, which is the extra stock we need to cover demand at that service level. We still multiply by the square root of the lead time to adjust for the number of days needed to restock. We then show the normal and non-parametric safety stock side by side to compare them.

In [ ]:
sku_stats["np_safety_stock_75"] = (sku_stats["q75_quantity"] - sku_stats["avg_quantity"]) * np.sqrt(sku_stats["avg_lead_time"])
sku_stats["np_safety_stock_90"] = (sku_stats["q90_quantity"] - sku_stats["avg_quantity"]) * np.sqrt(sku_stats["avg_lead_time"])
sku_stats["np_safety_stock_95"] = (sku_stats["q95_quantity"] - sku_stats["avg_quantity"]) * np.sqrt(sku_stats["avg_lead_time"])
sku_stats[["safety_stock_95", "np_safety_stock_95"]].head()

,safety_stock_95,np_safety_stock_95
sku_number,,
00D0DF3B,172.625460,169.387075
01083EC9,119.178354,135.273064
036A78DC,272.454716,210.902193
0469CB75,42.071814,48.884906
047A0938,175.247552,179.743245


The table shows that the two methods give similar safety stock for some SKUs but quite different results for others. For 00D0DF3B and 047A0938, the difference is small (172.6 vs 169.4 and 175.2 vs 179.7 units), so the normal assumption works almost as well for these items.

For 036A78DC, the normal formula gives a much higher safety stock than the non-parametric method (272.5 vs 210.9 units), so the company would keep about 62 units more than the real demand. For the smaller SKUs 01083EC9 and 0469CB75, it is the other way around: the normal formula gives less safety stock, so relying on this could lead to stockouts more often.

Normal assumption can either overestimate or underestimate the safety stock, depending on the SKU. The non-parametric safety stock doesn't assume that demand follows a normal distribution. It uses the actual order quantities of each SKU, so the real shape of each SKU's demand is reflected in the result.

### Comparison of the methods  

In [ ]:
sku_stats[["safety_stock_95", "np_safety_stock_95"]].describe()

,safety_stock_95,np_safety_stock_95
count,360.000000,360.000000
mean,102.025593,101.586381
std,78.827074,62.593608
min,35.022400,34.276871
25%,41.048831,47.837535
50%,49.145774,62.609748
75%,171.522346,169.550991
max,346.914705,265.456005


In [ ]:
sku_stats.sort_values("np_safety_stock_95", ascending = False).head(3)

,min_quantity,max_quantity,avg_quantity,median_quantity,var_quantity,std_quantity,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95,q75_quantity,q90_quantity,q95_quantity,np_safety_stock_75,np_safety_stock_90,np_safety_stock_95
sku_number,,,,,,,,,,,,,,,,
58023D4F,0,165,79.365556,91.0,1623.838683,40.296882,26.430544,139.733279,265.497588,340.762465,108.0,122.0,131.0,147.211521,219.186425,265.456005
A4287F10,0,171,81.038513,93.0,1696.831554,41.192615,26.215139,142.256070,270.290970,346.914705,110.0,123.0,132.0,148.284907,214.845845,260.926495
EAEE18E6,0,164,83.595689,96.0,1609.696055,40.121018,26.447866,139.169034,264.425507,339.386464,111.0,124.0,131.0,140.933493,207.789228,243.788471


In [ ]:
sku_stats.sort_values("np_safety_stock_95").head(3)

,min_quantity,max_quantity,avg_quantity,median_quantity,var_quantity,std_quantity,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95,q75_quantity,q90_quantity,q95_quantity,np_safety_stock_75,np_safety_stock_90,np_safety_stock_95
sku_number,,,,,,,,,,,,,,,,
BCBD9108,0,31,8.879121,9.0,37.440781,6.118887,14.0,15.442294,29.340841,37.658562,12.0,16.48,18.04,11.677260,28.439886,34.276871
F824ED29,0,23,10.142857,10.0,38.229323,6.182987,14.0,15.604062,29.648205,38.053059,15.0,18.88,20.00,18.173764,32.691395,36.882051
56DB54B0,0,35,8.925532,7.5,47.553535,6.895907,14.0,17.403267,33.066749,42.440713,14.0,17.00,18.89,18.986921,30.211893,37.283626


On average, the two methods give almost the same safety stock (about 102 units), but the rest of the describe table shows clear differences. The non-parametric safety stock is less spread out: its maximum is much lower (about 265 compared to 347 units), while its median and 25% values are higher. So the normal formula gives more safety stock to the SKUs at the top, and less to the smaller SKUs in the middle of the distribution.

The SKUs with the largest non-parametric safety stock are the same high-demand, high-variability SKUs with long lead times that had the largest normal safety stock, including A4287F10. So both methods agree on which products need the most safety stock, but the normal formula overestimates it. 

At the bottom, the smallest safety stock values are very similar for both methods, and the SKUs with the smallest non-parametric safety stock are again low-demand SKUs with stable orders and a 14-day lead time. However, they are not the same SKUs that had the smallest normal safety stock, which shows that the two methods can rank individual SKUs differently.

Overall, the normal formula seems to overestimate safety stock for the high-demand SKUs and underestimate it for many of the smaller ones, while the non-parametric method follows the real demand of each SKU.

## References  

- NIST/SEMATECH. (n.d.). *7.2.6.2. Percentiles*. In *NIST/SEMATECH e-Handbook of Statistical Methods*. National Institute of Standards and Technology. Retrieved November 18, 2026, from https://itl.nist.gov/div898/handbook/prc/section2/prc262.htm

- SciPy Developers. (n.d.-a). *scipy.stats.norm*. SciPy documentation. Retrieved November 14, 2026, from https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html

- SciPy Developers. (n.d.-b). *scipy.stats.mstats.mquantiles*. SciPy documentation. Retrieved November 18, 2026, from https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mstats.mquantiles.html

- Tutorials Point. (n.d.). *SciPy - stats.norm.ppf() function*. Retrieved November 14, 2026, from https://www.tutorialspoint.com/scipy/scipy_stats_norm_ppf_function.htm